##### (1) Response 대상 문장 추출 - ner_decision_results.review_candidate = true

In [ ]:
import pandas as pd
from src.config import get_postgre_connect

query = """
    SELECT DISTINCT data_version, press, doc_id
    FROM public.ner_decision_results
    WHERE review_candidate IS TRUE
"""

with get_postgre_connect() as conn:
    response_targets = pd.read_sql_query(query, conn)

C:\Users\PC2512\AppData\Local\Temp\ipykernel_6516\202509094.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  response_targets = pd.read_sql_query(query, conn)


In [4]:
response_targets

,data_version,press,doc_id
0,20261117_192958_강원일보,강원일보,NLRW2600000001.10006
1,20261117_192958_강원일보,강원일보,NLRW2600000001.10008
2,20261117_192958_강원일보,강원일보,NLRW2600000001.10014
3,20261117_192958_강원일보,강원일보,NLRW2600000001.1002
4,20261117_192958_강원일보,강원일보,NLRW2600000001.10024
...,...,...,...
136807,20261117_192337_충청일보,충청일보,NLRW2600000004.991
136808,20261117_192337_충청일보,충청일보,NLRW2600000004.9938
136809,20261117_192337_충청일보,충청일보,NLRW2600000004.9978
136810,20261117_192337_충청일보,충청일보,NLRW2600000004.9988


##### # (2) (1) 문장의 기사ID = 원본 데이터에서 전체 문장 추출 (기사 단위로 입력하기 위해) - ner_documents

In [ ]:
query = """
    WITH target_articles AS (
        SELECT DISTINCT data_version, press, doc_id
        FROM public.ner_decision_results
        WHERE review_candidate IS TRUE
        AND doc_id = 'NLRW2600000001.10006'
    )
    SELECT
        t.data_version, t.press, 
        n.doc_id, n.paragraph_id, n.sentence_id, n.form,
        n.paragraph_order, n.sentence_order,
        n.document_title, n.author, n.publisher, n.date, n.topic, n.original_topic,
        d.review_candidate, d.detected_entities, d.candidate_rule_ids,
        rule_evidence.rule_evidence,
        d.tagging_error_probability, d.tagging_error_result, d.tagging_error_confidence,
        d.tagging_merge_probability, d.tagging_merge_result, d.tagging_merge_confidence
    FROM target_articles AS t
    JOIN public.ner_documents AS n
      ON n.doc_id = t.doc_id
    LEFT JOIN public.ner_decision_results AS d
      ON d.data_version = t.data_version
     AND d.press = t.press
     AND d.doc_id = n.doc_id
     AND d.sentence_id = n.sentence_id
    LEFT JOIN LATERAL (
        SELECT jsonb_agg(
            jsonb_build_object(
                'rule_id', item.rule_id,
                'pattern', item.rule_result ->> 'pattern'
            ) ORDER BY item.rule_id
        ) AS rule_evidence
        FROM public.ner_rule_results AS rr
        CROSS JOIN LATERAL jsonb_each(rr.rule_results)
            AS item(rule_id, rule_result)
        WHERE rr.data_version = t.data_version
          AND rr.doc_id = n.doc_id
          AND rr.sentence_id = n.sentence_id
          AND (item.rule_result ->> 'detected')::boolean IS TRUE
    ) AS rule_evidence ON TRUE
    ORDER BY t.press, n.doc_id, n.sentence_order, n.sentence_id
"""

with get_postgre_connect() as conn:
    response_documents = pd.read_sql_query(query, conn)

C:\Users\PC2512\AppData\Local\Temp\ipykernel_6516\3883953567.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  response_documents = pd.read_sql_query(query, conn)


- response_documents 를 가지고 프롬프트 입력에 사용될 데이터 생성

In [ ]:
import json
import re

ENTITY_PATTERN = re.compile(r"<([^<>:]+):\s*(PS|LC|OG|AF|DT|EV)>")

def clean_value(value):
    return None if pd.isna(value) else value

def split_rule_ids(value):
    value = clean_value(value)
    return [item.strip() for item in str(value).split(",") if item.strip()] if value else []

def normalize_rule_evidence(value):
    value = clean_value(value)
    if isinstance(value, str):
        return json.loads(value) if value else []
    return value if isinstance(value, list) else []

documents = []
for (data_version, press, doc_id), doc_frame in response_documents.groupby(
    ["data_version", "press", "doc_id"], sort=False
):
    paragraphs = []
    for (paragraph_id, paragraph_order), paragraph_frame in doc_frame.groupby(
        ["paragraph_id", "paragraph_order"], sort=False, dropna=False
    ):
        sentences = []
        for _, row in paragraph_frame.sort_values("sentence_order").iterrows():
            sentence_id = row["sentence_id"]
            form = clean_value(row["form"]) or ""
            is_target = bool(clean_value(row["review_candidate"]) or False)
            rule_ids = split_rule_ids(row["candidate_rule_ids"])
            rule_evidence = normalize_rule_evidence(row["rule_evidence"])
            entities = [
                {
                    "entity_id": f"{sentence_id}.E{i}",
                    "target_span": match.group(0),
                    "target_type": match.group(2),
                    "rule_evidence": [],
                }
                for i, match in enumerate(ENTITY_PATTERN.finditer(form), start=1)
            ] if is_target else []
            decision = None
            if is_target:
                decision = {
                    "review_candidate": True,
                    "detected_entities": clean_value(row["detected_entities"]),
                    "candidate_rule_ids": rule_ids,
                    "tagging_error_probability": clean_value(row["tagging_error_probability"]),
                    "tagging_error_result": clean_value(row["tagging_error_result"]),
                    "tagging_error_confidence": clean_value(row["tagging_error_confidence"]),
                    "tagging_merge_probability": clean_value(row["tagging_merge_probability"]),
                    "tagging_merge_result": clean_value(row["tagging_merge_result"]),
                    "tagging_merge_confidence": clean_value(row["tagging_merge_confidence"]),
                }
            sentences.append({
                "id": sentence_id,
                "form": form,
                "entities": entities,
                "sentence_rule_evidence": rule_evidence,
                "decision": decision,
            })
        paragraphs.append({"id": paragraph_id, "sentence": sentences})

    first = doc_frame.iloc[0]
    documents.append({
        "id": doc_id,
        "metadata": {
            "title": clean_value(first["document_title"]),
            "author": clean_value(first["author"]),
            "publisher": press,
            "date": clean_value(first["date"]),
            "topic": clean_value(first["topic"]),
            "original_topic": clean_value(first["original_topic"]),
        },
        "paragraph": paragraphs,
    })

response_documents = {"document": documents}

##### (3) Response 배치 실행 테스트

- (3-1)프롬프트 정비
    - 태그오류와 태깅병합오류에 대한 내용이 decision에 이어 검증될수 있도록 해당 답변은 꼭 나와야함

- (3-2)api를 batch api로 수정

- (3-3)테스트 10건 진행 
    - 이때, batch를 선택하면 batch용으로 돌수있도록 고려해야함
    - openrouter를 사용할때도 고려해야함

- (3-4)결과 검증

##### (4) 언론사별 배치 진행
- 이전 run_decision_2에서 api 배치 진행한것처럼